In [ ]:
#!/usr/bin/env python3
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import LogLocator, LogFormatterMathtext
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import xarray as xr
import numpy as np
import pandas as pd

# ============================================================
# USER SETTINGS
# ============================================================
basedir = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/"
outdir  = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/"

# AMA extent
extent = (55, 115, -5, 40)  # (lonmin, lonmax, latmin, latmax)

# Threshold for "meaningful/non-zero" Δε coverage (h^-1)
tau = 0.5  # change to 0.1 if you prefer

# Output filenames
fig_png = outdir + "FigureS1_delta_metrics_summary.png"
fig_pdf = outdir + "FigureS1_delta_metrics_summary.pdf"
metrics_csv = outdir + "Figure_delta_metrics.csv"
metrics_png = outdir + "Figure_delta_metrics_summary.png"
metrics_pdf = outdir + "Figure_delta_metrics_summary.pdf"

# ============================================================
# INPUT FILES
# ============================================================
files = {
    "MCS CWP":      basedir + "Obs_cwp_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
    "MCS CIW":      basedir + "Obs_ciw_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
    "MCS CLW":      basedir + "Obs_clw_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
    "non-MCS CWP":  basedir + "Obs_cwp_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc",
    "non-MCS CIW":  basedir + "Obs_ciw_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc",
    "non-MCS CLW":  basedir + "Obs_clw_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc",
}

# ============================================================
# HELPERS
# ============================================================
def normalize_lon(da):
    """Shift lon to [-180, 180] if needed and sort."""
    if "lon" in da.coords and float(da.lon.max()) > 180:
        da = da.assign_coords(lon=((da.lon + 180) % 360) - 180).sortby("lon")
    return da

def subset_extent(da, ext):
    lonmin, lonmax, latmin, latmax = ext
    return da.sel(lon=slice(lonmin, lonmax), lat=slice(latmin, latmax))

def open_preceff(path, mask_for_log=False):
    """
    Read PRECEFF_TIMEAVG and convert to h^-1.
    If mask_for_log=True, mask non-positive values (for log plotting only).
    """
    da = xr.open_dataset(path)["PRECEFF_TIMEAVG"] * 3600.0  # -> h^-1
    da = normalize_lon(da)
    if mask_for_log:
        da = da.where(da > 0)
    return da

def area_weights(lat):
    """cos(lat) weights for area-weighting on a regular lat-lon grid."""
    return np.cos(np.deg2rad(lat))

def area_weighted_mean(da):
    """Area-weighted mean over lat/lon using cos(lat)."""
    w = xr.DataArray(area_weights(da.lat), coords={"lat": da.lat}, dims=("lat",))
    return da.weighted(w).mean(("lat", "lon"), skipna=True)

def area_fraction_condition(da, cond):
    """
    Area-weighted fraction of the domain where cond(da) is True.
    Uses cos(lat) weights and only counts finite da values in the denominator.
    """
    w = xr.DataArray(area_weights(da.lat), coords={"lat": da.lat}, dims=("lat",))
    finite = np.isfinite(da)
    mask = cond(da) & finite

    I = xr.where(mask, 1.0, 0.0)
    W = xr.where(finite, w, 0.0)  # broadcasts to lat,lon

    num = (I * W).sum(("lat", "lon"))
    den = W.sum(("lat", "lon"))
    return num / den

def add_map_pmesh(ax, da, title, cmap, norm, extent, show_left=False, show_bottom=False):
    im = ax.pcolormesh(
        da.lon, da.lat, da,
        cmap=cmap, norm=norm, shading="auto",
        transform=ccrs.PlateCarree(),
    )

    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.coastlines(linewidth=2.5)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=2.5)
    ax.add_feature(cfeature.LAKES.with_scale("50m"),
                   facecolor="none", edgecolor="0.5", linewidth=1)
    ax.add_feature(cfeature.LAND.with_scale("50m"),
                   facecolor="0.93", edgecolor="none", zorder=-10)

    gl = ax.gridlines(crs=ccrs.PlateCarree(), linewidth=1,
                      color="gray", alpha=0.4, linestyle="--",
                      draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = show_left
    gl.bottom_labels = show_bottom
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {"size": 35, "rotation": 35}
    gl.ylabel_style = {"size": 35, "rotation": 35}

    ax.set_title(title, fontsize=45, pad=6)
    im.set_rasterized(True)
    return im

# ============================================================
# LOAD DATA
#   - datasets_log: for rows 1–2 log plots (mask >0)
#   - datasets_raw: for Δ metrics & coverage stats (no mask)
# ============================================================
datasets_log = {k: open_preceff(v, mask_for_log=True)  for k, v in files.items()}
datasets_raw = {k: open_preceff(v, mask_for_log=False) for k, v in files.items()}

# subset both to AMA
for k in list(datasets_log.keys()):
    datasets_log[k] = subset_extent(datasets_log[k], extent)
    datasets_raw[k] = subset_extent(datasets_raw[k], extent)

# ============================================================
# DIFFERENCES
#   - diffs_plot: use log-ready datasets (will match what you plotted)
#   - diffs_raw: use raw datasets (best for rigorous coverage metrics)
# ============================================================
diffs_plot = {
    "CWP": datasets_log["MCS CWP"] - datasets_log["non-MCS CWP"],
    "CIW": datasets_log["MCS CIW"] - datasets_log["non-MCS CIW"],
    "CLW": datasets_log["MCS CLW"] - datasets_log["non-MCS CLW"],
}
diffs_raw = {
    "CWP": datasets_raw["MCS CWP"] - datasets_raw["non-MCS CWP"],
    "CIW": datasets_raw["MCS CIW"] - datasets_raw["non-MCS CIW"],
    "CLW": datasets_raw["MCS CLW"] - datasets_raw["non-MCS CLW"],
}

# ============================================================
# COLOR SCALES
# ============================================================
# Rows 1–2: LOG sequential (use datasets_log)
all_vals = xr.concat([datasets_log[k] for k in datasets_log], dim="panel").values
pos = np.isfinite(all_vals) & (all_vals > 0)

vmin_seq = np.nanpercentile(all_vals[pos], 1.0)
vmax_seq = np.nanpercentile(all_vals[pos], 99.5)
vmin_seq = max(vmin_seq, 1e-6)

norm_seq = mcolors.LogNorm(vmin=vmin_seq, vmax=vmax_seq)
cmap_seq = "gist_earth_r"

# Row 3: LINEAR diverging (use diffs_plot to match your plotted Δ)
all_diff = xr.concat([diffs_plot[k] for k in diffs_plot], dim="panel").values
valid_d = np.isfinite(all_diff)

vmax_diff = np.nanpercentile(np.abs(all_diff[valid_d]), 90.0)
vmin_diff = -vmax_diff

norm_diff = TwoSlopeNorm(vcenter=0.0, vmin=vmin_diff, vmax=vmax_diff)
cmap_diff = "coolwarm"

# ============================================================
# FIGURE (3x3 maps)
# ============================================================
proj = ccrs.PlateCarree()
fig, axes = plt.subplots(3, 3, figsize=(28, 18),
                         subplot_kw={"projection": proj})
fig.subplots_adjust(left=0.05, right=0.92,
                    bottom=0.06, top=0.95,
                    wspace=0.0, hspace=0.18)

titles_top = [r"$\bf{(a)}$ MCS $\epsilon_\mathrm{cwp}$",
              r"$\bf{(b)}$ MCS $\epsilon_\mathrm{i}$",
              r"$\bf{(c)}$ MCS $\epsilon_\ell$"]
titles_mid = [r"$\bf{(d)}$ non-MCS $\epsilon_\mathrm{cwp}$",
              r"$\bf{(e)}$ non-MCS $\epsilon_\mathrm{i}$",
              r"$\bf{(f)}$ non-MCS $\epsilon_\ell$"]
titles_bot = [r"$\bf{(g)}$ $\Delta$ $\epsilon_\mathrm{cwp}$",
              r"$\bf{(h)}$ $\Delta$ $\epsilon_i$",
              r"$\bf{(i)}$ $\Delta$ $\epsilon_\ell$"]

# Row 1: MCS (log)
for j, (k, t) in enumerate(zip(["MCS CWP", "MCS CIW", "MCS CLW"], titles_top)):
    im_seq = add_map_pmesh(axes[0, j], datasets_log[k], t,
                           cmap_seq, norm_seq, extent,
                           show_left=(j == 0), show_bottom=False)

# Row 2: non-MCS (log)
for j, (k, t) in enumerate(zip(["non-MCS CWP", "non-MCS CIW", "non-MCS CLW"], titles_mid)):
    im_seq = add_map_pmesh(axes[1, j], datasets_log[k], t,
                           cmap_seq, norm_seq, extent,
                           show_left=(j == 0), show_bottom=False)

# Row 3: Δ (linear diverging) — matches your original plot behavior
for j, (k, t) in enumerate(zip(["CWP", "CIW", "CLW"], titles_bot)):
    im_diff = add_map_pmesh(axes[2, j], diffs_plot[k], t,
                            cmap_diff, norm_diff, extent,
                            show_left=(j == 0), show_bottom=True)

# Colorbars
# LOG colorbar (rows 1–2)
cax1 = fig.add_axes([0.91, 0.41, 0.02, 0.50])
cb1 = fig.colorbar(im_seq, cax=cax1)
cb1.set_label(r"[h$^{-1}$]", fontsize=45)
cb1.ax.tick_params(labelsize=45)
cb1.locator = LogLocator(base=10)
cb1.formatter = LogFormatterMathtext()
cb1.update_ticks()

# Linear diverging colorbar (row 3)
cax2 = fig.add_axes([0.91, 0.08, 0.02, 0.23])
cb2 = fig.colorbar(im_diff, cax=cax2)
cb2.set_label(r"[h$^{-1}$]", fontsize=45)
cb2.ax.tick_params(labelsize=45)

fig.savefig(fig_png, dpi=200, bbox_inches="tight")
fig.savefig(fig_pdf, dpi=200, bbox_inches="tight")
plt.close(fig)

print(f"\nSaved maps:\n  {fig_png}\n  {fig_pdf}")

# ============================================================
# METRICS FROM Δ MAPS (use diffs_raw for rigor)
# ============================================================
rows = []
for key in ["CWP", "CIW", "CLW"]:
    d = diffs_raw[key]

    mean_delta = float(area_weighted_mean(d).values)
    mean_abs   = float(area_weighted_mean(np.abs(d)).values)

    frac_pos = float(area_fraction_condition(d, lambda x: x >  tau).values)
    frac_neg = float(area_fraction_condition(d, lambda x: x < -tau).values)
    frac_nz  = float(area_fraction_condition(d, lambda x: np.abs(x) > tau).values)

    rows.append({
        "pathway": key,
        "tau_h-1": tau,
        "mean_delta_h-1": mean_delta,
        "mean_abs_delta_h-1": mean_abs,
        "area_frac_|Δ|>tau": frac_nz,
        "area_frac_Δ>tau": frac_pos,
        "area_frac_Δ<-tau": frac_neg
    })

metrics = pd.DataFrame(rows).set_index("pathway")
print("\n=== Δε metrics (area-weighted over AMA; using RAW fields) ===")
print(metrics)

metrics.to_csv(metrics_csv)
print(f"\nSaved metrics table:\n  {metrics_csv}")

# ============================================================
# SUMMARY PLOT (bar chart): mean Δε + area fraction |Δ|>tau
# ============================================================
COLOR_CWP = "#6A5ACD"  # purple
COLOR_CIW = "#00A7B7"  # teal
COLOR_CLW = "#FFD60A"  # yellow
#

order = ["CWP", "CIW", "CLW"]
x = np.arange(len(order))

mean_delta = metrics.loc[order, "mean_delta_h-1"].values
frac_nz    = metrics.loc[order, "area_frac_|Δ|>tau"].values

fig2, ax = plt.subplots(figsize=(10, 6))

# mean Δε on left y-axis
ax.bar(x - 0.15, mean_delta, width=0.30,
       label=r"Area-weighted mean $\overline{\Delta\epsilon}$")
ax.set_ylabel(r"$\overline{\Delta\epsilon}$ [h$^{-1}$]", fontsize=25)
ax.axhline(0, linewidth=1)

# area fraction on right y-axis
ax2 = ax.twinx()
ax2.bar(x + 0.15, frac_nz, width=0.30, alpha=0.8,
        label=rf"Area fraction $|\Delta\epsilon|>{tau}$")
ax2.set_ylabel("Area fraction", fontsize=25)
ax2.set_ylim(0, 1)
ax2.tick_params(axis='y', labelsize=20)

xtick_labels = [
    r"$\epsilon_\mathrm{cwp}$",
    r"$\epsilon_i$",
    r"$\epsilon_\ell$",
]

#ax.set_xticks(x)
#ax.set_xticklabels(xtick_labels, fontsize=25)

# ---- X-axis formatting with colors
ax.set_xticks(x)
ax.set_xticklabels(xtick_labels, fontsize=25)

ax.tick_params(axis='y', labelsize=20)

#ax.set_title(rf"Δε (MCS − non-MCS); threshold $\tau={tau}$ h$^{{-1}}$", fontsize=25)

# Combined legend
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=13, frameon=False)

fig2.tight_layout()
fig2.savefig(metrics_png, dpi=200, bbox_inches="tight")
fig2.savefig(metrics_pdf, dpi=200, bbox_inches="tight")
#plt.close(fig2)
#plt.show()

print(f"\nSaved summary plot:\n  {metrics_png}\n  {metrics_pdf}")

# ============================================================
# OPTIONAL: print a ready-to-paste manuscript sentence template
# ============================================================
ciw = metrics.loc["CIW", "mean_delta_h-1"]
clw = metrics.loc["CLW", "mean_delta_h-1"]
fciw = metrics.loc["CIW", "area_frac_|Δ|>tau"]
fclw = metrics.loc["CLW", "area_frac_|Δ|>tau"]

print("\nManuscript-ready template (fill into text with these numbers):")
print(
    rf"Area-weighted mean enhancements over the AMA indicate a larger MCS–nonMCS contrast for ice-phase efficiency "
    rf"than liquid-phase efficiency ($\overline{{\Delta\epsilon_i}}={ciw:.2f}$ vs. "
    rf"$\overline{{\Delta\epsilon_\ell}}={clw:.2f}$ h$^{{-1}}$), and the spatial coverage of meaningful differences "
    rf"is greater for $\epsilon_i$ than for $\epsilon_\ell$ "
    rf"($f(|\Delta\epsilon_i|>\tau)={fciw:.2f}$ vs. $f(|\Delta\epsilon_\ell|>\tau)={fclw:.2f}$, $\tau={tau}$ h$^{{-1}}$)."
)
plt.show()